**Stream Splitting: Splitting a Methane Stream into Two Outlets**

In this tutorial we split a pure methane stream into two outlet streams using the IDAES `Separator` (splitter) unit model with the Peng-Robinson equation of state, and validate the result against an equivalent DWSIM Splitter model (Peng-Robinson property package).

We assume that the splitter operates at steady state. A splitter only divides the flow: every outlet has the same temperature, pressure and composition as the inlet.

### Inlet Specifications:

* **Component**: 100 % methane
* **Mass Flow Rate**: 3600 kg/h (1 kg/s)
* **Pressure**: 1.01325 bar
* **Temperature**: 25 °C

### Splitter Specifications:

* **Split basis**: total flow (`SplittingType.totalFlow`, DWSIM "Split Ratios")
* **Outlet 1**: 0.75 of the inlet (2700 kg/h)
* **Outlet 2**: 0.25 of the inlet (900 kg/h)

Only the split fraction of `outlet_1` is fixed; IDAES calculates `outlet_2` from the constraint that the split fractions add up to 1.

In [ ]:
import idaes

print("IDAES version: ", idaes.__version__)

## Setting up and solving the flowsheet

The flowsheet is Feed → Separator → two Products, built in `separator.py`. Degrees of freedom are printed before and after fixing the feed conditions and split fraction.

In [ ]:
from separator import setup_separator_model, report_separator_properties

# Create and solve the model
model, results = setup_separator_model(
    flow_mass=1.0,  # kg/s (3600 kg/h)
    pressure=101325,  # Pa
    temperature=25.0,  # C
    split_fraction=0.75,  # fraction to outlet_1
)
print(results.solver.termination_condition)

prop = report_separator_properties(model)

In [ ]:
# Visualize (if needed)
model.fs.visualize("Separator Model")

## DWSIM model

In [ ]:
from IPython.display import Image
Image(filename="DWSim Splitter.png")

In [ ]:
import webbrowser
import tempfile
import os

def display_image_in_browser(image_path, title="Image Viewer"):
    # Create a temporary HTML file that embeds the image
    html_content = f"""
    <html>
        <head>
            <title>{title}</title>
        </head>
        <body style="text-align: center;">
            <h2>{title}</h2>
            <img src="file://{os.path.abspath(image_path)}" style="max-width:90%; max-height:90%;">
        </body>
    </html>
    """
    with tempfile.NamedTemporaryFile('w', delete=False, suffix='.html') as f:
        f.write(html_content)
        temp_html_path = f.name

    webbrowser.open(f'file://{temp_html_path}')


In [ ]:
display_image_in_browser("DWSim Splitter.png", title="DWSim Splitter Model Check")

## Validation against DWSIM

The DWSIM results are read directly from the solved `.dwxmz` file and compared with IDAES. A splitter is a pure mass balance, so the results should match exactly.

In [ ]:
from separator import read_dwsim_results, compare_with_dwsim

dwsim = read_dwsim_results("DWSim Splitter Model.dwxmz")
compare_with_dwsim(model, dwsim)